In [1]:
# install dependencies
!pip install --upgrade pip
!pip install torch torchvision --index-url https://pytorch.org
!pip install -r requirements_multicontrolnet.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pytorch.org


In [2]:
import os
import gc
import torch
import random
import numpy as np
import cv2
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

In [3]:
!nvidia-smi

Tue Sep 15 00:35:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

Mounted at /content/drive


In [5]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/generative_rockets/

total 24043
drwx------ 2 root root    4096 Aug 19 21:16 .
drwx------ 4 root root    4096 Sep 15 00:36 ..
-rw------- 1 root root  191003 Aug 19 21:11 10.png
-rw------- 1 root root 1074311 Aug 22 22:18 11.png
-rw------- 1 root root  574114 Aug 22 22:20 12.png
-rw------- 1 root root  505267 Aug 22 22:22 13.png
-rw------- 1 root root  606461 Aug 22 22:23 14.png
-rw------- 1 root root  226947 Aug 22 22:23 15.png
-rw------- 1 root root  825329 Aug 22 22:25 16.png
-rw------- 1 root root  230436 Aug 24 15:54 17.png
-rw------- 1 root root  432220 Aug 24 15:55 18.png
-rw------- 1 root root  959075 Aug 24 15:56 19.png
-rw------- 1 root root 2071496 Aug 19 21:02 1.png
-rw------- 1 root root  658673 Aug 24 15:57 20.png
-rw------- 1 root root 1990696 Aug 24 15:57 21.png
-rw------- 1 root root  809303 Aug 24 15:57 22.png
-rw------- 1 root root  356262 Aug 24 15:58 23.png
-rw------- 1 root root  393754 Aug 24 16:00 24.png
-rw------- 1 root root  880891 Aug 24 16:34 25.png
-rw------- 1 root root  30026

In [6]:
# To make sure memory is empty
if "pipeline" in globals():
    del pipeline
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [8]:
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)

# SDLX base engine with ControlNet routing
pipeline = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

# VRAM saving tools
pipeline.vae.to(dtype=torch.float16)
pipeline.vae.enable_slicing()
pipeline.vae.enable_tiling()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 5.00GB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


...

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [20]:
image_dir = "/content/drive/MyDrive/generative_rockets/" 
image_paths = random.sample([os.path.join(image_dir, f) for f in sorted(os.listdir(image_dir)) if f.endswith(('png', 'jpg', 'jpeg'))],7)
print(image_paths)

composite_canvas = Image.open(image_paths[0]).convert("RGB").resize((1024, 1024))

for image in image_paths:
    layer = Image.open(image).convert("RGB").resize((1024,1024))
    composite_canvas = Image.blend(composite_canvas, layer, alpha=0.4) #Blends the structure of the images into one

img_np = np.array(composite_canvas)

# Detects sharp lines and structures
edges = cv2.Canny(img_np, 70, 150)
edges = edges[:,:,None]
edges = np.concatenate([edges, edges, edges], axis=2) #Get is back to RGB
unified_blueprint = Image.fromarray(edges)
# Saves the blue print
unified_blueprint.save("structural_blueprint_grid.png")

['/content/drive/MyDrive/generative_rockets/5.png', '/content/drive/MyDrive/generative_rockets/22.png', '/content/drive/MyDrive/generative_rockets/9.png', '/content/drive/MyDrive/generative_rockets/15.png', '/content/drive/MyDrive/generative_rockets/23.png', '/content/drive/MyDrive/generative_rockets/6.png', '/content/drive/MyDrive/generative_rockets/11.png']


In [21]:
# controlnet_conditioning_scale (0.0 to 1.0): 
# 1.0 means the AI must strictly stick to the blueprint lines.
# 0.5 means the AI can ignore minor lines to make the ship look more cohesive.
STRUCTURE_STRICTNESS = 0.50

positive_prompt = (
    "A unified, highly detailed sci-fi explorer spaceship, complex mechanical panelling, "
    "incorporating structural geometric shapes from the reference grid, make sure it's in space, 8k resolution,"
    "use blue in the engines and yellow in the windows"
)

negative_prompt = (
    "blurry, messy lines, text, watermark, bad proportions, floating debris, organic skin"
)

generator = torch.Generator(device="cuda").manual_seed(42)

output = pipeline(
    prompt=positive_prompt,
    negative_prompt=negative_prompt,
    image=unified_blueprint,
    controlnet_conditioning_scale=STRUCTURE_STRICTNESS,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator
)

output.images[0].save("controlnet_spaceship.png")
print("Image generated and saved")

  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


Image generated and saved
